# 03 – Fe-N4 DMET + SQD Catalysis Demo

Simulates a minimal Fe-N4 single-atom catalyst model using:

```
PySCF DFT(PBE)/def2-SVP  →  DMET embedding (Fe 3d active space)
                          →  Parity mapper
                          →  SQD solver (qiskit-addon-sqd)
                          ← 1-RDM feedback → DMET self-consistency
```

This demo:
1. Runs DFT on the full Fe-N4 cluster.
2. Applies DMET to isolate the Fe 3d fragment.
3. Solves with SQD (falls back to VQE if `qiskit-addon-sqd` not installed).
4. Compares energies: DFT vs DMET+SQD vs DMET+FCI reference.
5. Visualises the Fe fragment 1-RDM to show d-orbital occupations.

**Note on geometry**: The coordinates here are a toy 5-atom model.
For a realistic study use a DFT-optimised Fe-N4 porphyrin/SAC geometry.

In [ ]:
import os, sys, pathlib
_here = pathlib.Path().resolve()
for _root in [_here, _here.parent, _here.parent.parent]:
    if (_root / "pyproject.toml").is_file() and (_root / "dft_qc_pipeline").is_dir():
        sys.path.insert(0, str(_root))
        break

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

import numpy as np
import matplotlib.pyplot as plt

_NB = pathlib.Path().resolve()
_CFG_DIR = _NB.parent / "configs"

from dft_qc_pipeline import Pipeline, PipelineConfig
from dft_qc_pipeline.postprocessing import print_rdm1_summary


## 1. Load the Fe-N4 DMET+SQD config

In [ ]:
config = PipelineConfig.from_yaml(_CFG_DIR / "fen4_dmet_sqd.yaml")

print('Backend method:', config.backend.method.upper(), '/', config.backend.xc)
print('Basis:         ', config.backend.basis)
print('Embedding:     ', config.embedding.type)
print('Solver:        ', config.solver.type)
print()
for r in config.regions:
    print(f'  Region "{r.name}": atoms={r.atom_indices}, '
          f'nelec={r.nelec}, norb={r.norb}, loc={r.localization}')

# CI / nbmake: avoid SQD sampling + long DMET on shared runners
if os.environ.get("CI_FAST_NB", "").strip():
    config.embedding.max_iter = 1
    config.solver.type = "numpy"


## 2. Run the full DMET + SQD pipeline

In [ ]:
pipeline = Pipeline(config)
result = pipeline.run()

dft_energy  = result.backend_result.energy_hf
frag_energy = result.total_energy

print(f'DFT energy:               {dft_energy:.8f} Ha')
print(f'DMET+SQD fragment energy: {frag_energy:.8f} Ha')


## 3. Compare with FCI reference (DMET + NumPy)

In [ ]:
config_fci = PipelineConfig.from_yaml(_CFG_DIR / "fen4_dmet_sqd.yaml")
config_fci.solver.type = 'numpy'
config_fci.embedding.max_iter = 1  # quick one-shot for comparison

result_fci = Pipeline(config_fci).run()
fci_energy = result_fci.total_energy

print(f'DMET+FCI reference:       {fci_energy:.8f} Ha')
print(f'DMET+SQD energy:          {frag_energy:.8f} Ha')
print(f'Error SQD vs FCI:         {(frag_energy - fci_energy)*1000:.4f} mHa')


## 4. Spin-state comparison: S=0 vs S=2 vs S=4 (High-spin)

DFT often picks the wrong spin state for Fe-N4.  Here we check all three.

In [ ]:
spin_results = {}

spin_list = [0] if os.environ.get("CI_FAST_NB", "").strip() else [0, 2, 4]
for spin in spin_list:  # 2S
    cfg = PipelineConfig.from_yaml(_CFG_DIR / "fen4_dmet_sqd.yaml")
    cfg.backend.spin = spin
    cfg.solver.type = 'numpy'   # use FCI for clean comparison
    cfg.embedding.max_iter = 1
    try:
        r = Pipeline(cfg).run()
        spin_results[spin] = r.total_energy
        print(f'2S={spin}: E = {r.total_energy:.8f} Ha')
    except Exception as e:
        spin_results[spin] = float('nan')
        print(f'2S={spin}: FAILED ({e})')


In [ ]:
spins = list(spin_results.keys())
energies = [spin_results[s] for s in spins]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([f'2S={s}' for s in spins], energies,
              color=['#2196F3', '#FF9800', '#F44336'])
ax.set_ylabel('DMET+FCI fragment energy (Ha)')
ax.set_title('Fe-N4 spin-state energetics')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('FeN4_spin_states.png', dpi=150)
plt.show()

gs_spin = spins[np.argmin(energies)]
print(f'Ground state: 2S={gs_spin}')


## 5. Visualise Fe 3d-orbital occupations from 1-RDM

In [ ]:
frag_res = result_fci.fragment_results.get('Fe_d_active')
if frag_res and frag_res.rdm1 is not None:
    rdm1 = frag_res.rdm1
    print_rdm1_summary(rdm1, label='Fe 3d 1-RDM (DMET+FCI)')

    occupations = np.diag(rdm1)
    labels = [f'd_{i+1}' for i in range(len(occupations))]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(labels, occupations, color='#9C27B0', alpha=0.8)
    ax.axhline(1.0, color='k', ls='--', lw=1, label='half-filled')
    ax.axhline(2.0, color='r', ls='--', lw=1, label='fully occupied')
    ax.set_ylabel('Orbital occupation')
    ax.set_ylim(0, 2.2)
    ax.set_title('Fe 3d orbital occupations (DMET+FCI 1-RDM)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('FeN4_d_occupations.png', dpi=150)
    plt.show()
else:
    print('No 1-RDM available from this solver run.')


## 6. Swap embedding method: DMET → AVAS → ProjectorEmbedding

Same geometry, three embedding methods, compare fragment energies.

In [ ]:
embedding_results = {}

_emb_types = (
    ["dmet"]
    if os.environ.get("CI_FAST_NB", "").strip()
    else ["simple_cas", "avas", "dmet"]
)
for emb_type in _emb_types:
    cfg = PipelineConfig.from_yaml(_CFG_DIR / "fen4_dmet_sqd.yaml")
    cfg.embedding.type = emb_type
    cfg.embedding.max_iter = 1  # one-shot for speed
    cfg.solver.type = 'numpy'
    try:
        r = Pipeline(cfg).run()
        embedding_results[emb_type] = r.total_energy
        print(f'{emb_type:<15}: E = {r.total_energy:.8f} Ha')
    except Exception as e:
        embedding_results[emb_type] = float('nan')
        print(f'{emb_type:<15}: FAILED ({e})')
